In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from itertools import combinations, product
import matplotlib.pyplot as plt
import seaborn as sns
import copy
import os

from tqdm import tqdm
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CopulaGANSynthesizer, TVAESynthesizer, CTGANSynthesizer
from sklearn.metrics import r2_score
from scipy.interpolate import CubicSpline, PchipInterpolator, Akima1DInterpolator

import torch
import torch.nn as nn
import torch.optim as optim
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.api import ExponentialSmoothing


device = torch.device('cpu')

# plt.style.use('ggplot')
plt.style.use('default')

import matplotlib as mpl
#set params for the article
mpl.rcParams['xtick.labelsize'] = 16
mpl.rcParams['ytick.labelsize'] = 16
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.labelsize'] = 18

#set params for the notebook
# mpl.rcParams['xtick.labelsize'] = 12
# mpl.rcParams['ytick.labelsize'] = 12
# mpl.rcParams['legend.fontsize'] = 10
# mpl.rcParams['axes.labelsize'] = 14


# import category_encoders as ce

In [75]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

In [49]:
data = pd.read_csv('Data/CarPrice_Assignment.csv')
data = data.dropna()
data = data.drop(['CarName', 'car_ID', 'symboling'], axis=1)

X, y = data.drop('price', axis=1), data['price']
numeric_columns = X.select_dtypes(include=[np.number]).columns.tolist()
cat_ord_columns = ['doornumber']
cat_nominal_columns = np.setdiff1d(np.setdiff1d(X.columns, numeric_columns), cat_ord_columns)

X['doornumber'] = X['doornumber'].replace({'two': 2, 'four': 4})

/var/folders/31/d4jscthx2pn2pzww22f7p67r0000gn/T/ipykernel_82831/4056950033.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X['doornumber'] = X['doornumber'].replace({'two': 2, 'four': 4})


In [51]:
assert (len(numeric_columns) + len(cat_ord_columns) + len(cat_nominal_columns)) == len(X.columns)

In [63]:
X[cat_nominal_columns].apply(lambda x: len(x.unique()), axis=0).values.tolist()

[2, 5, 7, 3, 2, 7, 8, 2]

In [65]:
length_cat_ordinal = len(cat_ord_columns)
length_cat_onehot = X[cat_nominal_columns].apply(lambda x: len(x.unique()), axis=0).values.tolist()
length_num = len(cat_nominal_columns)

In [69]:
pd.concat([X[numeric_columns], y], axis=1)

,wheelbase,carlength,carwidth,carheight,curbweight,enginesize,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,88.6,168.8,64.1,48.8,2548,130,3.47,2.68,9.0,111,5000,21,27,13495.0
1,88.6,168.8,64.1,48.8,2548,130,3.47,2.68,9.0,111,5000,21,27,16500.0
2,94.5,171.2,65.5,52.4,2823,152,2.68,3.47,9.0,154,5000,19,26,16500.0
3,99.8,176.6,66.2,54.3,2337,109,3.19,3.40,10.0,102,5500,24,30,13950.0
4,99.4,176.6,66.4,54.3,2824,136,3.19,3.40,8.0,115,5500,18,22,17450.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,109.1,188.8,68.9,55.5,2952,141,3.78,3.15,9.5,114,5400,23,28,16845.0
201,109.1,188.8,68.8,55.5,3049,141,3.78,3.15,8.7,160,5300,19,25,19045.0
202,109.1,188.8,68.9,55.5,3012,173,3.58,2.87,8.8,134,5500,18,23,21485.0
203,109.1,188.8,68.9,55.5,3217,145,3.01,3.40,23.0,106,4800,26,27,22470.0


In [ ]:
scaler = MinMaxScaler()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=55555)

xgb = XGBRegressor()

xgb.fit(X_train, y_train)

/var/folders/31/d4jscthx2pn2pzww22f7p67r0000gn/T/ipykernel_82831/1725318317.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['doornumber'] = data['doornumber'].replace({'two': 2, 'four': 4})


In [81]:
label_enc_dict = {'encoders': []}

for col in cat_ord_columns:
    label_encoder = LabelEncoder()
    X[col] = label_encoder.fit_transform(X[col])
    
    label_enc_dict['encoders'].append(label_encoder)

In [82]:
label_enc_dict

{'encoders': [LabelEncoder()]}

In [83]:
X

,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,carheight,...,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg
0,gas,std,0,convertible,rwd,front,88.6,168.8,64.1,48.8,...,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27
1,gas,std,0,convertible,rwd,front,88.6,168.8,64.1,48.8,...,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27
2,gas,std,0,hatchback,rwd,front,94.5,171.2,65.5,52.4,...,six,152,mpfi,2.68,3.47,9.0,154,5000,19,26
3,gas,std,1,sedan,fwd,front,99.8,176.6,66.2,54.3,...,four,109,mpfi,3.19,3.40,10.0,102,5500,24,30
4,gas,std,1,sedan,4wd,front,99.4,176.6,66.4,54.3,...,five,136,mpfi,3.19,3.40,8.0,115,5500,18,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,gas,std,1,sedan,rwd,front,109.1,188.8,68.9,55.5,...,four,141,mpfi,3.78,3.15,9.5,114,5400,23,28
201,gas,turbo,1,sedan,rwd,front,109.1,188.8,68.8,55.5,...,four,141,mpfi,3.78,3.15,8.7,160,5300,19,25
202,gas,std,1,sedan,rwd,front,109.1,188.8,68.9,55.5,...,six,173,mpfi,3.58,2.87,8.8,134,5500,18,23
203,diesel,turbo,1,sedan,rwd,front,109.1,188.8,68.9,55.5,...,six,145,idi,3.01,3.40,23.0,106,4800,26,27


In [95]:
pd.get_dummies(X[cat_nominal_columns]).astype(float)

,aspiration_std,aspiration_turbo,carbody_convertible,carbody_hardtop,carbody_hatchback,carbody_sedan,carbody_wagon,cylindernumber_eight,cylindernumber_five,cylindernumber_four,...,fuelsystem_1bbl,fuelsystem_2bbl,fuelsystem_4bbl,fuelsystem_idi,fuelsystem_mfi,fuelsystem_mpfi,fuelsystem_spdi,fuelsystem_spfi,fueltype_diesel,fueltype_gas
0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
201,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
202,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
203,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [96]:
for col in cat_nominal_columns:
    print(pd.get_dummies(X[col]).astype(float))

     std  turbo
0    1.0    0.0
1    1.0    0.0
2    1.0    0.0
3    1.0    0.0
4    1.0    0.0
..   ...    ...
200  1.0    0.0
201  0.0    1.0
202  1.0    0.0
203  0.0    1.0
204  0.0    1.0

[205 rows x 2 columns]
     convertible  hardtop  hatchback  sedan  wagon
0            1.0      0.0        0.0    0.0    0.0
1            1.0      0.0        0.0    0.0    0.0
2            0.0      0.0        1.0    0.0    0.0
3            0.0      0.0        0.0    1.0    0.0
4            0.0      0.0        0.0    1.0    0.0
..           ...      ...        ...    ...    ...
200          0.0      0.0        0.0    1.0    0.0
201          0.0      0.0        0.0    1.0    0.0
202          0.0      0.0        0.0    1.0    0.0
203          0.0      0.0        0.0    1.0    0.0
204          0.0      0.0        0.0    1.0    0.0

[205 rows x 5 columns]
     eight  five  four  six  three  twelve  two
0      0.0   0.0   1.0  0.0    0.0     0.0  0.0
1      0.0   0.0   1.0  0.0    0.0     0.0  0.0
2   